In [13]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI


load_dotenv()


model = ChatGoogleGenerativeAI(model="gemini-3.6-flash")


#State
class BlogState(TypedDict):
    title:str
    outline:str
    content:str
    evaluation:int

#function definition
def create_outline(state:BlogState)->BlogState:
    #fetch title
    title=state['title']

    #call LLM outline
    prompt=f'Generate a detailed outline for a blog on the topic-{title}'
    outline=model.invoke(prompt).content
    #update state
    state['outline']=outline
    return state

def create_blog(state:BlogState)->BlogState:
    #fetch state
    title=state['title']
    outline=state['outline']

    #call to llm
    prompt=f'Write a detailed blog on the title -{title} using the following outline-{outline}'
    content=model.invoke(prompt).content
    state['content']=content
    return state

def evaluate(state:BlogState)->BlogState:
    #fetch state
    content=state['content']
    outline=state['outline']

    #call to llm
    prompt=f'Based on this {outline} evaluate my blog {content} in integer score out of 10 (e.g., "Score: 8/10").'
    evaluation=model.invoke(prompt).content
    state['evaluation']=evaluation
    return state


#graph
graph=StateGraph(BlogState)

#nodes
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('evaluate',evaluate)

#edges
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog','evaluate')
graph.add_edge('evaluate',END)


#compile
workflow=graph.compile()

#Show Grapgh
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())
#run
initial_state={'title':'Rise of AI in Pakistan'}
final_state=workflow.invoke(initial_state)

print(final_state['outline'])


GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}